#### ***Pydantic***
- Pydantic is the most widely used data validation and settings management library for Python, utilizing type hints to ensure data structure integrity. 
- It validates, parses, and coerces data at runtime to match specified types, making it essential for building robust APIs and handling external data, with its core logic written in Rust for high performance.

In [2]:
## Problem without Pydantic 
def add_patient(name:str,age:int):
    if type(name) == str and type(age) == int:
        if age >= 0:
            print(name)
            print(age)
            print("Data added successfully to the database!")
        else:
            raise ValueError("Age cannot be negative.")
    else:
        raise TypeError("Invalid data type for name or age. Name should be a string and age should be an integer.")

In [3]:
def update_patient(name:str,age:int):
    if type(name) == str and type(age) == int:
        if age >= 0:
            print(name)
            print(age)
            print("Data updated successfully in the database!")
        else:
            raise ValueError("Age cannot be negative.")
    else:
        raise TypeError("Invalid data type for name or age. Name should be a string and age should be an integer.")

In [4]:
add_patient("Alex",22)

Alex
22
Data added successfully to the database!


#### ***Usage of Pydantic***
- Define a Pydantic model that represents the ideal schema of the data.
- This includes the expected fields, their types, and any validation constraints (e.g., gt=0 for positive numbers).
- Instantiate the model with raw input data (usually a dictionary or JSON-like structure).
- Pydantic will automatically validate the data and coerce it into the correct Python types (if possible).
- If the data doesn't meet the model's requirements, Pydantic raises a ValidationError.
- Pass the validated model object to functions or use it throughout your codebase.
- This ensures that every part of your program works with clean, type-safe, and logically valid data.

In [5]:
from pydantic import BaseModel 

class Patient(BaseModel):
    name:str
    age:int 
    weight:float

def add_data(patient:Patient):
    print(patient.name)
    print(patient.age)
    print(patient.weight)
    print("data added success..!")

def update_data(patient:Patient):
    print(patient.name)
    print(patient.age)
    print(patient.weight)
    print("data updated success..!")

patient_data = {"name": "Alex", "age": "25", "weight": "70.5"}
p1 = Patient(**patient_data)
add_data(p1)

Alex
25
70.5
data added success..!


- ***Little Complex***

In [6]:
from pydantic import BaseModel 
from typing import List,Dict

class Patient(BaseModel):
    name:str
    age:int 
    weight:float
    married:bool 
    allergies:List[str]
    contact_info:Dict[str,str]

def add_data(patient:Patient):
    print(patient.name)
    print(patient.age)
    print(patient.weight)
    print("data added success..!")

def update_data(patient:Patient):
    print(patient.name)
    print(patient.age)
    print(patient.weight)
    print("data updated success..!")

- ***Required & Optional Fields***

In [ ]:
# Required & Optional Field

from pydantic import BaseModel
from typing import List,Dict,Optional

class PatientData(BaseModel):
    name:str
    age:int 
    weight:float
    married:bool = False 
    allergies:Optional[List[str]] = None 
    contact_info:Dict[str,str]

def add_data(patient:PatientData):
    print(patient.name)
    print(patient.age)
    print(patient.weight)
    print(patient.married)
    print(patient.allergies)
    print("Data added successfully..!")

In [8]:
patient_data = {"name": "Bappy", "age": "25", "weight": "70.5", "contact_info": {"email": "bappy@example.com", "phone": "123-456-7890"}}

p1= PatientData(**patient_data)
add_data(p1)

Bappy
25
70.5
False
None
Data added successfully..!


In [9]:
from pydantic import BaseModel,EmailStr,AnyUrl,Field
from typing import List,Dict,Optional

class EmployeeData(BaseModel):
    name: str = Field(max_length=50)
    email: EmailStr
    linkedin_url: AnyUrl
    age: int = Field(gt=0, lt=100)
    weight: float
    married: bool = False
    allergies: Optional[List[str]] = Field(max_length=5)
    contact_info: Dict[str, str]

def add_employee(emp: EmployeeData):
    print(emp.name)
    print(emp.age)
    print(emp.weight)
    print(emp.married)
    print(emp.allergies)
    print("Data added successfully to the database...")

emp_data = {"name": "Bappy", "email": "bappy@gmail.com", 
                "linkedin_url": "https://www.linkedin.com/in/boktiarahmed73/", 
                "age": "25", "weight": "70.5", 
                "allergies": ["peanuts", "shellfish"],
                "contact_info": {"phone": "123-456-7890"}}

e1 = EmployeeData(**emp_data)
add_employee(e1)

Bappy
25
70.5
False
['peanuts', 'shellfish']
Data added successfully to the database...


#### ***Field Validator***

In [ ]:
# Field Validator

from pydantic import BaseModel, EmailStr,field_validator
from typing import List, Dict

class PersonData(BaseModel):
    name: str
    email: EmailStr
    age: int
    weight: float
    married: bool
    allergies: List[str]
    contact_details: Dict[str, str]

    @field_validator('email')
    @classmethod
    def email_validator(cls,value):
        valid_domain = ['hdfc.com', 'icici.com','gmail.com']
        domain_name = value.split('@')[-1]
        if domain_name not in valid_domain:
                raise ValueError('not a valid domain')
        return value
    
    @field_validator('name')
    @classmethod
    def transform_name(cls,value):
        return value.upper()
    

def add_person(p:PersonData):
    print(p.name)
    print(p.age)
    print(p.weight)
    print(p.married)
    print(p.allergies)
    print("Data added successfully to the database..!")

data = {"name": "bappy", "email": "bappy@hdfc.com", 
                "linkedin_url": "https://www.linkedin.com/in/boktiarahmed73/", 
                "age": "25", "weight": 70.5, 
                "married": "True",
                "allergies": ["peanuts", "shellfish"],
                "contact_details": {"phone": "123-456-7890"}}

alex = PersonData(**data)
add_person(alex)

BAPPY
25
70.5
True
['peanuts', 'shellfish']
Data added successfully to the database..!


#### ***Model Validator(handle logical condition)***


In [ ]:
# Model Validator

from pydantic import BaseModel, EmailStr, AnyUrl, Field, field_validator, model_validator
from typing import List, Dict, Optional, Annotated

class EmpData(BaseModel):
    name: str
    email: EmailStr
    age: int
    weight: float
    married: bool
    allergies: List[str]
    contact_details: Dict[str, str]

    @model_validator(mode='after')
    def valid_contact(cls,model):
        if model.age > 60 and 'emergency' not in model.contact_details:
            raise ValueError('retitred employee')
        return model

def add_data(emp:EmpData):
    print(emp.name)
    print(emp.age)
    print(emp.weight)
    print(emp.married)
    print(emp.allergies)
    print("Data added successfully to the database!")

data = {"name": "Ankit", "email": "bappy@hdfc.com", 
                "linkedin_url": "https://www.linkedin.com/in/boktiarahmed73/", 
                "age": "70", "weight": 70.5, 
                "married": "True",
                "allergies": ["peanuts", "shellfish"],
                "contact_details": {"phone": "123-456-7890", "emergency": "987-654-3210"}}

emp1 = EmpData(**data)
add_data(emp1)

Ankit
70
70.5
True
['peanuts', 'shellfish']
Data added successfully to the database!


C:\Users\gzbra\AppData\Local\Temp\ipykernel_10804\2590901230.py:13: PydanticDeprecatedSince212: Using `@model_validator` with mode='after' on a classmethod is deprecated. Instead, use an instance method. See the documentation at https://docs.pydantic.dev/2.13/concepts/validators/#model-after-validator. Deprecated in Pydantic V2.12 to be removed in V3.0.
  @model_validator(mode='after')


#### ***Computed Filed***
- Allow creating read-only properties derived dynamically from other fields.

In [ ]:
# Computed Field

from pydantic import BaseModel, EmailStr, AnyUrl, Field, field_validator, model_validator, computed_field
from typing import List, Dict, Optional, Annotated

class PatientData(BaseModel):

    name: str
    email: EmailStr
    age: int
    weight: float
    height: float
    married: bool
    allergies: List[str]
    contact_details: Dict[str, str]


    @computed_field
    @property
    def bmi(self) -> float:
        bmi = round(self.weight/(self.height**2),2)
        return bmi



def add_patient_data(patient: PatientData):
    print(patient.name)
    print(patient.age)
    print(patient.weight)
    print(patient.married)
    print(patient.allergies)
    print("BMI:", patient.bmi)
    print("Data added successfully to the database!")




patient_data = {"name": "bappy", "email": "bappy@hdfc.com", 
                "linkedin_url": "https://www.linkedin.com/in/boktiarahmed73/", 
                "age": "70", "weight": 70.5, 
                "height": 1.75,
                "married": "True",
                "allergies": ["peanuts", "shellfish"],
                "contact_details": {"phone": "123-456-7890", "emergency": "987-654-3210"}}

patient_1 = PatientData(**patient_data)

add_patient_data(patient_1)

bappy
70
70.5
True
['peanuts', 'shellfish']
BMI: 23.02
Data added successfully to the database!


#### ***Nested Model***

In [14]:
from  pydantic import BaseModel

class Address(BaseModel):
    city:str 
    state:str
    pin:str

class PatientData(BaseModel):
    name:str 
    gender:str 
    age:int  
    address:Address

address_dict = {'city': 'gurgaon', 'state': 'haryana', 'pin': '122001'}
address = Address(**address_dict)

patient_dict = {'name': 'Kishor', 'gender': 'male', 'age': 40, 'address': address}
patient = PatientData(**patient_dict)

print(patient)
print(patient.name)
print(patient.address)
print(patient.address.city)

name='Kishor' gender='male' age=40 address=Address(city='gurgaon', state='haryana', pin='122001')
Kishor
city='gurgaon' state='haryana' pin='122001'
gurgaon


#### ***Serialization***

In [ ]:
# Serialization

from  pydantic import BaseModel
class Address(BaseModel):
    city:str 
    state:str 
    pin:str

class PatientInfo(BaseModel):
    name: str
    gender: str
    age: int
    address: Address

address_dict = {'city': 'gurgaon', 'state': 'haryana', 'pin': '122001'}
address = Address(**address_dict)

patient_dict = {'name': 'Kishor', 'gender': 'male', 'age': 40, 'address': address}
patient = PatientInfo(**patient_dict)

temp = patient.model_dump()
print(temp)
print(type(temp))

{'name': 'Kishor', 'gender': 'male', 'age': 40, 'address': {'city': 'gurgaon', 'state': 'haryana', 'pin': '122001'}}
<class 'dict'>
